In [ ]:
#!unzip /content/data.zip
!rm -r /content/data/Meta-Llama-3-8B-Instruct/

In [ ]:
from pathlib import Path
import pandas as pd
import re
import io

# ====== EDIT THESE ======
DATA_DIR = Path("/content/data")          # Data folder (contains LLM folders)
OUT_CSV  = Path("/content/master_responses.csv")
# ========================

PROMPT_COL = "Prompt (Disability Context)"

DISABILITY_COLS = {
    "Response (Disability Context Vision Impairments)": "Vision Impairments",
    "Response (Disability Context Hearing Impairments)": "Hearing Impairments",
    "Response (Disability Context Speech Impairments)": "Speech Impairments",
    "Response (Disability Context Mobility Impairments)": "Mobility Impairments",
    "Response (Disability Context Neurological Disorders)": "Neurological Disorders",
    "Response (Disability Context Genetic and Developmental Disorders)": "Genetic and Developmental Disorders",
    "Response (Disability Context Learning Disorders)": "Learning Disorders",
    "Response (Disability Context Sensory Processing and Cognitive Disorders)": "Sensory Processing and Cognitive Disorders",
    "Response (Disability Context Mental Health and Behavioral Disorders)": "Mental Health and Behavioral Disorders",
}

PLACEHOLDER_RE = re.compile(r"\{disability\}", re.IGNORECASE)

def norm_col(c: str) -> str:
    c = str(c).strip()
    c = re.sub(r"\s+", " ", c)
    return c

def read_csv_robust(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            pass
    raw = path.read_bytes()
    text = raw.decode("latin1", errors="replace")
    return pd.read_csv(io.StringIO(text))

def find_prompt_col(cols):
    cols = list(cols)
    if PROMPT_COL in cols:
        return PROMPT_COL
    for c in cols:
        if "Prompt" in c and "Disability Context" in c:
            return c
    raise KeyError(f"Prompt column not found. Expected '{PROMPT_COL}'.")

def find_disability_cols(cols):
    cols_set = set(cols)
    if set(DISABILITY_COLS.keys()).issubset(cols_set):
        return DISABILITY_COLS  # exact match preferred

    found = {}
    for c in cols:
        m = re.match(r"Response\s*\(Disability Context\s*(.*?)\)\s*$", c)
        if m:
            found[c] = m.group(1).strip()

    if len(found) < 9:
        raise KeyError(f"Found only {len(found)} response columns. Columns: {cols}")
    return found

def replace_disability_placeholder(text: str, disability_type: str) -> str:
    if text is None:
        return ""
    text = str(text)
    # Replace {disability} (case-insensitive) with the disability type string
    return PLACEHOLDER_RE.sub(disability_type, text)

def build_master_from_data_dir(data_dir: Path, out_csv: Path) -> pd.DataFrame:
    if not data_dir.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {data_dir}")

    llm_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()])
    if not llm_dirs:
        raise FileNotFoundError(f"No subfolders found inside: {data_dir}")

    print(f"Found {len(llm_dirs)} LLM folders under {data_dir}")

    rows = []
    missing_files = []

    for llm_dir in llm_dirs:
        llm_name = llm_dir.name

        candidates = list(llm_dir.glob("education_srikant.csv"))
        if not candidates:
            candidates = [p for p in llm_dir.glob("*.csv") if "education_srikant" in p.name.lower()]

        if not candidates:
            missing_files.append(llm_name)
            continue

        csv_path = sorted(candidates)[0]
        df = read_csv_robust(csv_path)
        df.columns = [norm_col(c) for c in df.columns]

        prompt_col = find_prompt_col(df.columns)
        resp_map = find_disability_cols(df.columns)

        df = df.reset_index(drop=True)

        for i, r in df.iterrows():
            question_id = i + 1
            base_query = "" if pd.isna(r.get(prompt_col, "")) else str(r.get(prompt_col)).strip()

            for resp_col, disability_type in resp_map.items():
                # Replace {disability} with the disability type for THIS row
                query_text = replace_disability_placeholder(base_query, disability_type)

                resp = r.get(resp_col, "")
                response_text = "" if pd.isna(resp) else str(resp).strip()

                response_id = f"{llm_name}__Q{question_id:02d}__{disability_type}"

                rows.append({
                    "response_id": response_id,
                    "llm_name": llm_name,
                    "disability_type": disability_type,
                    "question_id": question_id,
                    "query_text": query_text,
                    "response_text": response_text,
                })

    master = pd.DataFrame(rows)

    # Add numeric id 1..N in a stable order
    master = master.sort_values(["llm_name", "question_id", "disability_type"]).reset_index(drop=True)
    master.insert(0, "response_num", range(1, len(master) + 1))

    print("\nMaster summary:")
    print("Rows:", len(master))
    if not master.empty:
        print("LLMs:", master["llm_name"].nunique())
        print("Questions:", master["question_id"].nunique())
        print("Disabilities:", master["disability_type"].nunique())

    if missing_files:
        print("\nWARNING: No education_srikant.csv found in these LLM folders:")
        for name in missing_files:
            print(" -", name)

    # If you expect exactly 16*38*9 = 5472, this will show whether you got it
    expected = master["llm_name"].nunique() * master["question_id"].nunique() * master["disability_type"].nunique()
    if len(master) != expected:
        print(f"\nWARNING: Row count {len(master)} != expected {expected}. "
              f"(This can happen if some files have fewer questions or missing LLM folders.)")

    master.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")

    return master

# Run
master_df = build_master_from_data_dir(DATA_DIR, OUT_CSV)


Found 16 LLM folders under /content/data

Master summary:
Rows: 5472
LLMs: 16
Questions: 38
Disabilities: 9

Saved: /content/master_responses.csv


In [ ]:
!unzip /content/rating.zip


In [ ]:
from pathlib import Path
import pandas as pd
import json
import re

MASTER_CSV = Path("/content/master_responses.csv")
RATING_DIR = Path("/content/rating")
OUT_CSV    = Path("/content/master_responses_with_tokens.csv")

# ✅ Correct label groups (keys become your new column prefixes)
LABEL_GROUPS = {
    "relevance_general":       ["relevance_general"],
    "support_specific":        ["support_specific"],
    "stigmatizing_language":   ["stigmatizing_language"],
    "accessibility_conflict":  ["accessibility_conflict"],
    "answer_education_request":["answer_education_request"],
}

TOKEN_COLS = {k: f"{k}_tokens" for k in LABEL_GROUPS}
SCORE_COLS = {k: f"{k}_confidence_score" for k in LABEL_GROUPS}

def norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip())

def safe_read_json(path: Path) -> dict:
    raw = path.read_bytes()
    try:
        return json.loads(raw.decode("utf-8"))
    except UnicodeDecodeError:
        return json.loads(raw.decode("latin1", errors="replace"))

def load_ratings_map(rating_dir: Path):
    rating_map = {}
    for llm_dir in rating_dir.iterdir():
        if not llm_dir.is_dir():
            continue
        llm_name = llm_dir.name
        for jf in llm_dir.glob("*.json"):
            data = safe_read_json(jf)
            qn = data.get("question_number")
            dis = data.get("disability_type")
            tokens = data.get("tokens", [])
            if qn is None or dis is None:
                continue
            key = (llm_name, int(qn), norm_space(dis))
            rating_map[key] = tokens
    return rating_map

def extract_lists(tokens_list):
    out = {k: ([], []) for k in LABEL_GROUPS}  # tokens, scores
    for t in tokens_list or []:
        tok = t.get("token", "")
        lbl = t.get("label", "")
        score = t.get("score", None)
        if tok is None or str(tok).strip() == "":
            continue
        for group_key, allowed in LABEL_GROUPS.items():
            if lbl in allowed:
                out[group_key][0].append(str(tok))
                out[group_key][1].append(score if score is not None else None)
                break
    return out

# Load master + normalize disability for matching
master = pd.read_csv(MASTER_CSV)
master["__dis_norm"] = master["disability_type"].astype(str).apply(norm_space)

# Load all rating jsons
rating_map = load_ratings_map(RATING_DIR)

# Add empty columns
for k in LABEL_GROUPS:
    master[TOKEN_COLS[k]] = "[]"
    master[SCORE_COLS[k]] = "[]"

missing = 0
for i, row in master.iterrows():
    key = (str(row["llm_name"]), int(row["question_id"]), row["__dis_norm"])
    tokens_list = rating_map.get(key)
    if tokens_list is None:
        missing += 1
        continue
    lists = extract_lists(tokens_list)
    for k, (tok_list, score_list) in lists.items():
        master.at[i, TOKEN_COLS[k]] = json.dumps(tok_list, ensure_ascii=False)
        master.at[i, SCORE_COLS[k]] = json.dumps(score_list, ensure_ascii=False)

master.drop(columns=["__dis_norm"], inplace=True)

print("Missing JSON matches:", missing, "out of", len(master))
master.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)


Missing JSON matches: 1 out of 5472
Saved: /content/master_responses_with_tokens.csv


In [ ]:
import pandas as pd, json, numpy as np
from pathlib import Path

IN_CSV  = Path("/content/master_responses_with_tokens.csv")
OUT_CSV = Path("/content/sampled_for_manual_eval.csv")

SEED = 42
N_PER_STRATUM = 137   # 137*4 = 548

REL  = "relevance_general_tokens"
SUP  = "support_specific_tokens"
ANS  = "answer_education_request_tokens"
STIG = "stigmatizing_language_tokens"
CONF = "accessibility_conflict_tokens"

def parse_list(x):
    if pd.isna(x): return []
    s = str(x).strip()
    if s == "" or s.lower() == "none": return []
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else []
    except Exception:
        return []

df = pd.read_csv(IN_CSV)

# counts
df["rel_n"]  = df[REL].apply(lambda z: len(parse_list(z)))
df["sup_n"]  = df[SUP].apply(lambda z: len(parse_list(z)))
df["ans_n"]  = df[ANS].apply(lambda z: len(parse_list(z)))
df["stig_n"] = df[STIG].apply(lambda z: len(parse_list(z)))
df["conf_n"] = df[CONF].apply(lambda z: len(parse_list(z)))

# flags
df["harm"] = ((df["stig_n"] > 0) | (df["conf_n"] > 0)).astype(int)
df["helpful_count"] = df["rel_n"] + df["sup_n"] + df["ans_n"]
med = df["helpful_count"].median()
df["helpful"] = (df["helpful_count"] >= med).astype(int)

print("Available per stratum:")
print(df.groupby(["harm","helpful"]).size())

rng = np.random.default_rng(SEED)
parts = []
for h in [0,1]:
    for hp in [0,1]:
        sub = df[(df.harm==h) & (df.helpful==hp)]
        if len(sub) < N_PER_STRATUM:
            raise ValueError(f"Stratum harm={h}, helpful={hp} has only {len(sub)} rows (<{N_PER_STRATUM}).")
        idx = rng.choice(sub.index.to_numpy(), size=N_PER_STRATUM, replace=False)
        parts.append(df.loc[idx])

sampled = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
sampled["stratum"] = "harm=" + sampled["harm"].astype(str) + ", helpful=" + sampled["helpful"].astype(str)

keep = ["response_num","response_id","llm_name","disability_type","question_id","query_text","response_text",
        "harm","helpful","helpful_count","stratum"]
sampled[keep].to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print("Sample counts:")
print(sampled.groupby(["harm","helpful"]).size())


2) Create assignment files for 6 authors (exactly your overlap plan)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

IN_CSV = Path("/content/sampled_for_manual_eval.csv")  # 548 rows
OUT_ALL = Path("/content/manual_eval_assignments.csv")
OUT_DIR = Path("/content/assignments_by_author")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
OVERLAP_N = 110
AUTHORS = ["A0","A1","A2","A3","A4","A5"]
OTHERS = ["A1","A2","A3","A4","A5"]

df = pd.read_csv(IN_CSV).copy()
rng = np.random.default_rng(SEED)

# pick overlap set
overlap_idx = rng.choice(df.index.to_numpy(), size=OVERLAP_N, replace=False)
df["is_overlap"] = False
df.loc[overlap_idx, "is_overlap"] = True

assign_rows = []

# A0 gets ALL overlap (110)
for _, r in df[df["is_overlap"]].iterrows():
    d = r.to_dict()
    d["assigned_to"] = "A0"
    assign_rows.append(d)

# Split overlap into ~22 each for A1..A5
overlap_df = df[df["is_overlap"]].sample(frac=1, random_state=SEED).reset_index(drop=True)
chunks = np.array_split(overlap_df, len(OTHERS))
for a, chunk in zip(OTHERS, chunks):
    for _, r in chunk.iterrows():
        d = r.to_dict()
        d["assigned_to"] = a
        assign_rows.append(d)

# Remaining 438 single-coded -> distribute across A1..A5
single_df = df[~df["is_overlap"]].sample(frac=1, random_state=SEED).reset_index(drop=True)
for j, (_, r) in enumerate(single_df.iterrows()):
    a = OTHERS[j % len(OTHERS)]
    d = r.to_dict()
    d["assigned_to"] = a
    assign_rows.append(d)

assigned = pd.DataFrame(assign_rows)

# Save combined + per-author
assigned.to_csv(OUT_ALL, index=False)
print("Saved:", OUT_ALL)

for a in AUTHORS:
    sub = assigned[assigned["assigned_to"] == a].copy()
    out = OUT_DIR / f"{a}_items.csv"
    sub.to_csv(out, index=False)
    print(a, "items:", len(sub), "->", out)


# To add colmun fo metric

In [ ]:
import pandas as pd
import json
from pathlib import Path

ASSIGN_CSV = Path("/content/manual_eval_assignments.csv")
MASTER_CSV = Path("/content/master_responses_with_tokens.csv")
OUT_CSV    = Path("/content/manual_eval_assignments_with_metrics.csv")

REL  = "relevance_general_tokens"
SUP  = "support_specific_tokens"
ANS  = "answer_education_request_tokens"
STIG = "stigmatizing_language_tokens"
CONF = "accessibility_conflict_tokens"

HELP_METRICS = [
    ("relevance_general", REL),
    ("support_specific", SUP),
    ("answer_education_request", ANS),
]
HARM_METRICS = [
    ("stigmatizing_language", STIG),
    ("accessibility_conflict", CONF),
]

def parse_list(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return []
    try:
        v = json.loads(s)
        return v if isinstance(v, list) else []
    except Exception:
        # if it's already a python-like list string or malformed, fail safe
        return []

# ---- load files ----
assign = pd.read_csv(ASSIGN_CSV)
master = pd.read_csv(MASTER_CSV)

# ensure response_num is numeric for a clean join
assign["response_num"] = pd.to_numeric(assign["response_num"], errors="coerce")
master["response_num"] = pd.to_numeric(master["response_num"], errors="coerce")

# ---- compute metrics presence on master (one time) ----
master["rel_n"]  = master[REL].apply(lambda z: len(parse_list(z)))
master["sup_n"]  = master[SUP].apply(lambda z: len(parse_list(z)))
master["ans_n"]  = master[ANS].apply(lambda z: len(parse_list(z)))
master["stig_n"] = master[STIG].apply(lambda z: len(parse_list(z)))
master["conf_n"] = master[CONF].apply(lambda z: len(parse_list(z)))

def present_metrics(row, metrics):
    present = []
    for name, col in metrics:
        if len(parse_list(row[col])) > 0:
            present.append(name)
    return ";".join(present)

master["harm_metrics_present"] = master.apply(lambda r: present_metrics(r, HARM_METRICS), axis=1)
master["help_metrics_present"] = master.apply(lambda r: present_metrics(r, HELP_METRICS), axis=1)

# keep only lookup columns for merge (prevents duplicating big text fields)
lookup_cols = [
    "response_num",
    "harm_metrics_present",
    "help_metrics_present",
    "rel_n","sup_n","ans_n","stig_n","conf_n",
]
lookup = master[lookup_cols].copy()

# ---- merge onto your assignment file ----
out = assign.merge(lookup, on="response_num", how="left")

out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
print("Rows with missing lookup:", out["harm_metrics_present"].isna().sum(), "out of", len(out))


In [ ]:
import pandas as pd
from pathlib import Path

FILE = Path("/content/Manual validation_  Helpful or Harmful_ A Token-Level Evaluation of LLM Responses to Accessibility-Oriented Educational Queries  (Responses).xlsx")

ANNOT = "annotator_id"
RID   = "response_num"

metrics = [
    "Relevance score",
    "Support score",
    "Answer score",
    "Conflict score",
    "Stigma score",
]

pairs = [("A0","A1"), ("A0","A2"), ("A0","A3")]   # change/add if needed

df = pd.read_excel(FILE)

# Keep only needed cols + drop rows missing key info
df = df[[RID, ANNOT] + metrics].copy()
df = df.dropna(subset=[RID, ANNOT])

# Ensure response_num is consistent
df[RID] = pd.to_numeric(df[RID], errors="coerce")
df = df.dropna(subset=[RID])
df[RID] = df[RID].astype(int)

pair_dfs = {}

for a0, aX in pairs:
    sub = df[df[ANNOT].isin([a0, aX])].copy()

    # keep only response_nums that have BOTH annotators
    both = sub.groupby(RID)[ANNOT].nunique()
    both_ids = both[both == 2].index
    sub = sub[sub[RID].isin(both_ids)]

    # pivot to get A0 and AX side-by-side for each metric
    out = pd.DataFrame({RID: sorted(both_ids)}).set_index(RID)

    for m in metrics:
        p = sub.pivot(index=RID, columns=ANNOT, values=m)
        # add two columns: metric__A0 and metric__AX
        out[f"{m}__{a0}"] = p[a0]
        out[f"{m}__{aX}"] = p[aX]

    out = out.reset_index()
    pair_dfs[f"{a0}_vs_{aX}"] = out

    # optional: save
    out.to_csv(f"/content/{a0}_vs_{aX}_paired_scores.csv", index=False)
    print(f"{a0}_vs_{aX}: rows={len(out)} saved to /content/{a0}_vs_{aX}_paired_scores.csv")

# If you want to inspect in notebook:
A0_vs_A1 = pair_dfs["A0_vs_A1"]
A0_vs_A2 = pair_dfs["A0_vs_A2"]
A0_vs_A3 = pair_dfs["A0_vs_A3"]


# ---- Function to compute agreement ----
def compute_matches(pair_df, pair_name):

    print(f"\n===== Agreement for {pair_name} =====")

    results = []

    for m in metrics:
        colA = f"{m}__A0"
        colB = [c for c in pair_df.columns if c.startswith(m) and c != colA][0]

        matches = (pair_df[colA] == pair_df[colB]).sum()
        total   = len(pair_df)
        pct     = 100 * matches / total if total else 0

        results.append([m, matches, total, pct])

    res_df = pd.DataFrame(results, columns=["Metric","Matches","Total","Percent"])
    print(res_df)
    return res_df


# ---- Run for each pair ----
res_A1 = compute_matches(A0_vs_A1, "A0 vs A1")
res_A2 = compute_matches(A0_vs_A2, "A0 vs A2")
res_A3 = compute_matches(A0_vs_A3, "A0 vs A3")



overall = pd.concat([res_A1,res_A3]) \
            .groupby("Metric")[["Matches","Total"]] \
            .sum()

overall["Percent"] = 100 * overall["Matches"] / overall["Total"]
print("\n===== Overall Agreement =====")
print(overall)



In [ ]:
import pandas as pd
from pathlib import Path

FILE = Path("/content/Manual validation_  Helpful or Harmful_ A Token-Level Evaluation of LLM Responses to Accessibility-Oriented Educational Queries  (Responses).xlsx")

ANNOT = "annotator_id"
RID   = "response_num"

metrics = [
    "Relevance score",
    "Support score",
    "Answer score",
    "Conflict score",
    "Stigma score",
]

df = pd.read_excel(FILE)

# Normalize stigma text
df["Stigma score"] = df["Stigma score"].replace({"No":0, "Yes":1})

df[RID] = pd.to_numeric(df[RID], errors="coerce")
df = df.dropna(subset=[RID])

# --------------------------------------------
results = {}

for m in metrics:

    accepted_values = []
    disagreements = 0

    for rid, group in df.groupby(RID):

        vals = group[m].dropna().tolist()

        if len(vals) == 1:
            accepted_values.append(vals[0])

        elif len(vals) == 2:
            if vals[0] == vals[1]:
                accepted_values.append(vals[0])
            else:
                disagreements += 1

    # distribution
    dist = pd.Series(accepted_values).value_counts().sort_index()

    results[m] = {
        "total_counted": len(accepted_values),
        "disagreements": disagreements,
        "distribution": dist
    }

# --------------------------------------------
# Print nicely
for m, res in results.items():
    print(f"\n===== {m} =====")
    print("Counted responses:", res["total_counted"])
    print("Discarded disagreements:", res["disagreements"])
    print("Value distribution:")
    print(res["distribution"])
